# TMC-LM on Google Colab

Trains the **TinyLlama-1.1B-Chat** LoRA for Trinidad Municipal College and outputs a GGUF
model for local Ollama use.

**Pipeline:** sources \u2192 dataset \u2192 LoRA fine-tune \u2192 merge \u2192 GGUF (Q4_K_M)

**Runtime:** Free tier T4 GPU. If you are on free tier, keep batch size 1 (default).

## 1. Mount Google Drive

Persistence lives under `MyDrive/tmc-llm/`:
- `.hf-cache/` - Hugging Face model cache (avoids re-downloading ~2.2 GB every session)
- `data/raw/tmc_sources/` - **put your source documents here** for future training runs
- `models/adapters`, `models/merged`, `models/gguf` - training outputs

Everything is reused if the session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

DRIVE_ROOT = "/content/drive/MyDrive/tmc-llm"
CACHE_DIR = f"{DRIVE_ROOT}/.hf-cache"
SOURCES_DIR = f"{DRIVE_ROOT}/data/raw/tmc_sources"
MODELS_DIR = f"{DRIVE_ROOT}/models"
ADAPTER_DIR = f"{MODELS_DIR}/adapters/tmc-lm-tinyllama-lora"
MERGED_DIR = f"{MODELS_DIR}/merged/tmc-lm-tinyllama"
GGUF_DIR = f"{MODELS_DIR}/gguf"

for d in [CACHE_DIR, SOURCES_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("Drive paths ready:")
print("  HF cache :", CACHE_DIR)
print("  Sources  :", SOURCES_DIR)
print("  Adapters :", ADAPTER_DIR)
print("  Merged   :", MERGED_DIR)
print("  GGUF     :", GGUF_DIR)

## 2. Clone the repo and install dependencies

The repo is cloned fresh each session (code updates get picked up automatically).
Your persistent source documents are copied in from Drive.**torch is already preinstalled in Colab**, so only the remaining libraries are installed.

In [ ]:
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/jbasilad/tmc-llm"
REPO_DIR = "/content/tmc-llm"

if not os.path.isdir(f"{REPO_DIR}/.git"):
    print(">> Cloning repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"

src_repo = pathlib.Path("data/raw/tmc_sources")
src_repo.mkdir(parents=True, exist_ok=True)
for item in pathlib.Path(SOURCES_DIR).iterdir():
    dst = src_repo / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(item, dst)

print("Repo ready at:", os.getcwd())

In [ ]:
def run_script(script):
    print(f">> Running {script}")
    subprocess.run(["/bin/bash", script], check=True)

print(">> Installing ML libraries (torch already present in Colab)")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "peft", "datasets", "accelerate", "sentencepiece",
    "PyMuPDF", "python-docx", "openpyxl", "pyyaml", "gguf",
], check=True)

print(">> Installing the tmc_llm package")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", device)

## 3. Prepare the dataset

Builds `data/processed/{dataset,train,validation,test}.jsonl` from the source documents in `data/raw/tmc_sources/`.

In [ ]:
os.environ["SOURCE_DIR"] = "data/raw/tmc_sources"
os.environ["OUTPUT_DIR"] = "data/processed"
run_script("scripts/prepare_dataset.sh")

In [ ]:
import json
metadata = json.load(open("data/processed/metadata.json", encoding="utf-8"))
print(f"Source files   : {len(metadata['source_files'])}")
print(f"Total examples : {metadata['total_examples']}")
print(f"Train examples : {metadata['train_examples']}")
print(f"Validation     : {metadata['validation_examples']}")
print(f"Test examples  : {metadata['test_examples']}")
print("Sections       :", ", ".join(metadata["sections"]))

## 4. Fine-tune TinyLlama with LoRA

Uses `configs/train_lora.yaml` hyperparameters (fp16, batch size 1, ~80 steps) but sends the
adapter to Drive so it survives session resets. On a free T4 this takes roughly 10-20 minutes.

In [ ]:
import pathlib
import yaml

cfg = yaml.safe_load(pathlib.Path("configs/train_lora.yaml").read_text(encoding="utf-8"))
cfg["output_dir"] = ADAPTER_DIR

colab_cfg = pathlib.Path("configs/train_lora_colab.yaml")
colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(colab_cfg.read_text(encoding="utf-8"))

In [ ]:
os.environ["CONFIG_PATH"] = "configs/train_lora_colab.yaml"
run_script("scripts/train_lora.sh")
print(">> Adapter saved to", ADAPTER_DIR)

## 5. Merge the LoRA adapter into the base model

Produces a full merged model in Drive under `models/merged/tmc-lm-tinyllama/`.

In [ ]:
os.environ["BASE_MODEL"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
os.environ["ADAPTER_DIR"] = ADAPTER_DIR
os.environ["OUTPUT_DIR"] = MERGED_DIR
run_script("scripts/merge_lora.sh")
print(">> Merged model saved to", MERGED_DIR)

## 6. Convert to GGUF (llama.cpp, built inside Colab)

No Docker needed. The script clones llama.cpp, builds only the `llama-quantize` binary
(CPU-only, portable), converts the merged model to F16, then quantizes to **Q4_K_M**.
First run takes a few extra minutes for the llama.cpp build.

In [ ]:
os.environ["MERGED_MODEL_DIR"] = MERGED_DIR
os.environ["OUTPUT_DIR"] = GGUF_DIR
os.environ["BUILD_JOBS"] = "2"
run_script("scripts/convert_to_gguf.sh")

In [ ]:
import glob, os
for f in sorted(glob.glob(f"{GGUF_DIR}/*.gguf")):
    print(f"{os.path.basename(f):<40} {os.path.getsize(f)/1e6:8.1f} MB")

## 7. Sanity check (in-notebook QA)

Quick test of the **merged** model against a few TMC knowledge questions before you download.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

qa_model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
qa_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

SYSTEM = (
    "You are TMC-LM, an offline assistant for Trinidad Municipal College. "
    "Answer using only official TMC knowledge. If the answer is not in the "
    "source, say that the available TMC source does not contain it. "
    "Be concise and professional."
)

def format_for_qa(tokenizer, messages):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    user = next(m["content"] for m in messages if m["role"] == "user")
    return f"### User: {user}\n### Assistant:"

for question in [
    "What is the vision of TMC?",
    "What academic programs does TMC offer?",
    "Give a brief history of TMC.",
]:
    prompt = format_for_qa(
        qa_tokenizer,
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
    )
    inputs = qa_tokenizer(prompt, return_tensors="pt").to(qa_model.device)
    outputs = qa_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
    )
    answer = qa_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {question}\nA: {answer.strip()}\n")

## 8. Next steps: use the model locally with Ollama

Your Q4_K_M GGUF is ready on Google Drive:

**`MyDrive/tmc-llm/models/gguf/tmc-lm-tinyllama-q4_k_m.gguf`**

On your Windows PC (inside the `tmc-llm` repo):
1. Download that file from Drive into `models/gguf/` (use the larger `-f16.gguf` too if you want a higher-quality variant).
2. Verify the file:
   `scripts/check_gguf.ps1`
3. Import into Ollama:
   `scripts/create_ollama_model.ps1`
4. Chat:
   `ollama run tmc-lm`

> **Session timeout:** free Colab sessions end after long idle periods. The notebook re-runs
> from scratch in seconds because models/cache live on Drive - just press **Run all** again.

In [ ]:
print("GGUF files on Drive:")
import glob, os
for f in sorted(glob.glob(f"{GGUF_DIR}/*.gguf")):
    print("  ", f, f"({os.path.getsize(f)/1e6:.1f} MB)")
print("\nDownload 'tmc-lm-tinyllama-q4_k_m.gguf' from Drive, then on Windows run:")
print("  .\\scripts\\create_ollama_model.ps1")
print("  ollama run tmc-lm")